# Spine Foundation Model — Colab Training

Trains the multi-task lumbar spine model on a free Colab GPU:

- **Landmark localization** — L1-L5 vertebrae + L1/L2…L5/S1 discs, per-point confidence
- **Disc degeneration (DDD)** grading — when `ddd_labels.csv` is provided
- **Spondylolisthesis** slip estimation — when `spondy_labels.csv` is provided
- Longitudinal clinical risk model — trained separately via `train_clinical.py`

**Before you start:** make sure your dataset is in Google Drive — a plain
folder works (default expected at `MyDrive/dataset`; edit `DATASET_DIR` in
cell 5 if it lives elsewhere). A `dataset.zip` also works.

In [ ]:
# 1. Check GPU
!nvidia-smi -L || echo 'Runtime > Change runtime type > GPU'

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone the repository (no weights needed — we train here)
%cd /content
import os
if not os.path.exists('/content/spine-foundation'):
    !git clone https://github.com/codermisba/spinelit-ai.git spine-foundation
%cd /content/spine-foundation

In [ ]:
# 4. Install dependencies (PyTorch is preinstalled on Colab)
!pip install -q timm gradio tqdm scikit-learn pandas
import torch
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 5. Load the dataset from Google Drive (folder OR zip)
import os, shutil

DRIVE_ROOT = '/content/drive/MyDrive'
REPO_DATASET = '/content/spine-foundation/dataset'

# >>> EDIT if your folder lives somewhere else in Drive <<<
DATASET_DIR = f'{DRIVE_ROOT}/dataset'
DATASET_ZIP = f'{DRIVE_ROOT}/spine-foundation/dataset.zip'

def merge_copy(src, dst):
    """Recursively copy src into dst without deleting anything."""
    count = 0
    for root, _, files in os.walk(src):
        target = os.path.normpath(
            os.path.join(dst, os.path.relpath(root, src))
        )
        os.makedirs(target, exist_ok=True)
        for name in files:
            shutil.copy2(os.path.join(root, name),
                         os.path.join(target, name))
            count += 1
    return count

if os.path.isdir(DATASET_DIR):
    print('Copying dataset folder from Drive...')
    print(f'  {merge_copy(DATASET_DIR, REPO_DATASET)} files copied.')
elif os.path.exists(DATASET_ZIP):
    !unzip -q -o "{DATASET_ZIP}" -d "{REPO_DATASET}"
else:
    print('Dataset not found at:', DATASET_DIR)
    print('\nYour MyDrive contains:')
    !ls "{DRIVE_ROOT}"
    raise FileNotFoundError(
        'Edit DATASET_DIR in this cell to point at your Drive folder.'
    )

print('\nTop level of dataset/:')
!ls "{REPO_DATASET}" | head -20

In [ ]:
# 6. GPU-friendly settings (edit config.py values instead if preferred)
import re
cfg = open('config.py').read()
cfg = cfg.replace('IMAGE_SIZE = 256', 'IMAGE_SIZE = 512')
cfg = cfg.replace('BATCH_SIZE = 4', 'BATCH_SIZE = 32')
cfg = cfg.replace('NUM_WORKERS = 0', 'NUM_WORKERS = 2')
open('config.py','w').write(cfg)
print('Config updated for GPU training.')

In [ ]:
# 7. Sanity checks
!python dataset.py && python model.py

In [ ]:
# 8. TRAIN (checkpoints land in checkpoints/)
#    Add labels CSVs before this step to also train DDD / spondylolisthesis.
!python train.py --epochs 60

In [ ]:
# 9. Evaluate on the validation split
!python evaluate.py

In [ ]:
# 10. Try one prediction
!python predict.py --image "$(find dataset/data -name '*.jpg' | head -1)" \
    --age 58 --sex female --pain-scale 6 --modality mri \
    --pain-years 4 --start-year 2022 --years-ahead 5

In [ ]:
# 11. Optional: train the longitudinal clinical risk model
#     Requires dataset/longitudinal_records.csv (schema in README)
!python train_clinical.py --epochs 150

In [ ]:
# 12. SAVE everything to Google Drive for further use
import shutil, os
DEST = '/content/drive/MyDrive/spine-foundation/checkpoints'
os.makedirs(DEST, exist_ok=True)
for f in ['best_model.pth', 'last_model.pth', 'longitudinal_model.pth']:
    src = f'checkpoints/{f}'
    if os.path.exists(src):
        shutil.copy2(src, DEST)
        print('saved', src, '->', DEST)
print('\nDone. Your trained models are safe in Drive at:', DEST)

In [ ]:
# 13. Optional: launch the UI right inside Colab
!python app.py --share